# Enriquecimiento cross-domain de tráfico de red con embeddings de comportamiento

- Objetivo: detectar tráfico de red anómalo enriqueciendo el dataset de tráfico con contexto de comportamiento de usuario, usando autoencoders adversariales para alinear ambos dominios sin correspondencia etiquetada.
- Este notebook acompaña al artículo completo, con el detalle de las decisiones de ingeniería y las tablas de condiciones/pasos: **artículo completo →** https://fuzzyfrog.ai/es/ai-lab/proyectos/ciberseguridad/enriquecimiento-cross-domain-embeddings-conductuales-deteccion-anomalias-red/
- **Nota sobre los datos:** el dataset de tráfico de red (`UNSW-NB15`, y como validación `CIC-IDS2017`) es público. El dataset de comportamiento de usuario es **100% sintético**, generado para este notebook con la misma estructura (frecuencia de uso por servicio, distribución por perfil) que un dataset original de comportamiento, pero sin ningún dato real de personas. Se incluye como `dataset_sintetico_perfiles_comportamiento.csv` en este repositorio.


## Diagrama del pipeline

El pipeline tiene tres etapas:

1. **Construcción de perfiles de comportamiento** — del dataset sintético a un embedding latente por usuario (autoencoder + clustering jerárquico).
2. **Enriquecimiento cross-domain** — dos autoencoders adversariales (uno por dominio) + un discriminador que fuerza a que ambos espacios latentes compartan geometría, seguido de un emparejamiento por similitud coseno instancia a instancia.
3. **Clasificación** — una red feedforward (DFNN) entrenada sobre el vector enriquecido (96D), comparada contra un baseline entrenado solo con features de tráfico.

El diagrama interactivo completo, con el detalle de cada bloque, está en el artículo (sección "Diagrama de la solución").

In [ ]:
# Diagrama simplificado del pipeline (representación estática, ver el diagrama interactivo en el artículo)
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

boxes = [
    (0.02, 0.55, 0.28, "Tráfico de red\n(público)"),
    (0.02, 0.15, 0.28, "Perfil de comportamiento\n(sintético)"),
    (0.36, 0.55, 0.28, "Autoencoder\ndominio tráfico"),
    (0.36, 0.15, 0.28, "Autoencoder\ndominio conductual"),
    (0.70, 0.35, 0.28, "Discriminador +\nembeddings alineados"),
]
for x, y, w, label in boxes:
    ax.add_patch(patches.FancyBboxPatch((x, y), w, 0.28, boxstyle="round,pad=0.02",
                                         linewidth=1.2, edgecolor="#006a87", facecolor="#f4f6f8"))
    ax.text(x + w/2, y + 0.14, label, ha="center", va="center", fontsize=9)

plt.title("Construcción de perfiles → enriquecimiento cross-domain → clasificación", fontsize=10)
plt.tight_layout()
plt.show()


## Carga de datos

- Se cargan dos fuentes: el dataset sintético de comportamiento y el dataset público de tráfico de red (UNSW-NB15).
- El dataset sintético reemplaza al dataset original de comportamiento por razones de confidencialidad; conserva la misma estructura (usuarios × frecuencia de uso por servicio) pero ningún valor es real.
- Ajusta `RUTA_UNSW` a la ubicación local de tu copia del dataset UNSW-NB15 (binario, normal/ataque).

In [ ]:
import pandas as pd
import numpy as np

# --- Comportamiento (sintético) ---
df_comportamiento = pd.read_csv("dataset_sintetico_perfiles_comportamiento.csv")
print("Perfiles de comportamiento (sintéticos):", df_comportamiento.shape)
df_comportamiento.head()


In [ ]:
# --- Tráfico de red (UNSW-NB15, público) ---
# Descarga: https://research.unsw.edu.au/projects/unsw-nb15-dataset
RUTA_UNSW = "unsw_nb15_binario.csv"  # ajustar a tu ruta local

df_trafico = pd.read_csv(RUTA_UNSW)
print("Tráfico de red (UNSW-NB15):", df_trafico.shape)
df_trafico.head()


## Explicación de los datos

- **Comportamiento (sintético):** cada fila es un usuario, cada columna un servicio monitoreado; el valor es el conteo de interacciones durante la ventana de observación. `perfil_origen_sintetico` es solo metadato de generación, no se usa como feature.
- **Tráfico (UNSW-NB15):** cada fila es un evento/flujo de red; se usa la versión binaria (`label`: 0 normal, 1 ataque), con las columnas identificantes (usuario, IP, categoría de ataque) removidas por diseño del dataset original.

In [ ]:
feature_cols_comportamiento = [c for c in df_comportamiento.columns
                                if c not in ["usuario_id", "perfil_origen_sintetico"]]
print(f"{len(feature_cols_comportamiento)} servicios monitoreados por usuario")

print("\nDistribución de label en tráfico:")
print(df_trafico["label"].value_counts(normalize=True))


## Análisis de datos / EDA

- Frecuencia normalizada de uso por servicio y entropía de Shannon por usuario, sobre el dataset sintético.
- Estas dos features, más 2 componentes de PCA, son la base del perfil de comportamiento antes del autoencoder.

In [ ]:
from sklearn.decomposition import PCA

X_comp = df_comportamiento[feature_cols_comportamiento].values.astype(float)
totales = X_comp.sum(axis=1, keepdims=True)
freq_norm = X_comp / totales

# Entropía de Shannon por usuario
eps = 1e-12
entropia = -(freq_norm * np.log(freq_norm + eps)).sum(axis=1)

pca = PCA(n_components=2, random_state=42)
pca_comp = pca.fit_transform(freq_norm)

df_features_comportamiento = pd.DataFrame(freq_norm, columns=feature_cols_comportamiento)
df_features_comportamiento["entropia"] = entropia
df_features_comportamiento["pca_1"] = pca_comp[:, 0]
df_features_comportamiento["pca_2"] = pca_comp[:, 1]

print("Varianza explicada por los 2 componentes:", pca.explained_variance_ratio_.sum().round(3))
df_features_comportamiento.describe().T[["mean", "std", "min", "max"]].round(3)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df_features_comportamiento["entropia"], bins=15, color="#006a87")
axes[0].set_title("Distribución de entropía de actividad")
axes[1].scatter(df_features_comportamiento["pca_1"], df_features_comportamiento["pca_2"],
                 c=df_comportamiento["perfil_origen_sintetico"].astype("category").cat.codes, cmap="viridis", s=25)
axes[1].set_title("PCA (2 componentes) de frecuencias de uso")
plt.tight_layout()
plt.show()


## Modelado

### 6.1 Perfilamiento conductual: autoencoder + clustering

- Se compara clustering directo (K-means, DBSCAN, jerárquico) sobre las features crudas contra un híbrido: autoencoder (reducción a 2D) + clustering jerárquico sobre el espacio latente.
- El híbrido gana consistentemente en silhouette score.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

feature_cols_finales = feature_cols_comportamiento + ["entropia", "pca_1", "pca_2"]
X = StandardScaler().fit_transform(df_features_comportamiento[feature_cols_finales].values)

resultados = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    resultados[("kmeans", k)] = silhouette_score(X, km.labels_)
    ag = AgglomerativeClustering(n_clusters=k).fit(X)
    resultados[("agglomerative", k)] = silhouette_score(X, ag.labels_)

mejor_directo = max(resultados.items(), key=lambda kv: kv[1])
print("Mejor silhouette sobre datos crudos:", mejor_directo)


In [ ]:
class AutoencoderConductual(nn.Module):
    def __init__(self, n_features, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, 8), nn.ReLU(), nn.Linear(8, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, n_features), nn.Sigmoid())

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

X_t = torch.tensor(X, dtype=torch.float32)
ae = AutoencoderConductual(n_features=X.shape[1], latent_dim=2)
opt = optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(100):
    opt.zero_grad()
    recon, _ = ae(X_t)
    loss = loss_fn(recon, torch.sigmoid(X_t))
    loss.backward()
    opt.step()

with torch.no_grad():
    _, latente = ae(X_t)
latente = latente.numpy()

mejor_hibrido = {}
for k in range(2, 8):
    ag = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(latente)
    mejor_hibrido[k] = silhouette_score(latente, ag.labels_)

k_optimo = max(mejor_hibrido, key=mejor_hibrido.get)
print("Silhouette por k (autoencoder + jerárquico):", {k: round(v, 3) for k, v in mejor_hibrido.items()})
print(f"Mejor configuración: k={k_optimo}, silhouette={mejor_hibrido[k_optimo]:.3f}")

etiquetas_finales = AgglomerativeClustering(n_clusters=k_optimo, linkage="ward").fit_predict(latente)
print("Davies-Bouldin:", round(davies_bouldin_score(latente, etiquetas_finales), 3))


### 6.2 Enriquecimiento cross-domain: autoencoders adversariales

- Un autoencoder por dominio (tráfico y comportamiento), cada uno a un embedding de 32D.
- Un discriminador entrenado para distinguir el dominio de origen de cada embedding; los autoencoders se entrenan también para engañarlo (peso adversarial 0.1 sobre el binary cross-entropy).
- 100 épocas, batch size 128, Adam con lr=1e-3.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Preparación de tráfico: features numéricas, imputación de faltantes/infinitos, estandarización
X_trafico = df_trafico.drop(columns=["label"]).select_dtypes(include=[np.number]).copy()
X_trafico = X_trafico.replace([np.inf, -np.inf], np.nan).fillna(X_trafico.median(numeric_only=True))
X_trafico_scaled = StandardScaler().fit_transform(X_trafico.values)
y_trafico = df_trafico["label"].values

X_comportamiento_scaled = X  # ya estandarizado arriba

class Autoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, 64), nn.ReLU(), nn.Linear(64, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(),
                                      nn.Linear(64, n_features), nn.Sigmoid())

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

class Discriminador(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, z):
        return self.net(z)

ae_trafico = Autoencoder(n_features=X_trafico_scaled.shape[1], latent_dim=32)
ae_comportamiento = Autoencoder(n_features=X_comportamiento_scaled.shape[1], latent_dim=32)
discriminador = Discriminador(latent_dim=32)

opt_ae = optim.Adam(list(ae_trafico.parameters()) + list(ae_comportamiento.parameters()), lr=1e-3)
opt_disc = optim.Adam(discriminador.parameters(), lr=1e-3)
mse = nn.MSELoss()
bce = nn.BCELoss()

Xt_trafico = torch.tensor(X_trafico_scaled, dtype=torch.float32)
Xt_comportamiento = torch.tensor(X_comportamiento_scaled, dtype=torch.float32)

PESO_ADVERSARIAL = 0.1
EPOCAS = 100

for epoch in range(EPOCAS):
    # --- Paso del discriminador ---
    opt_disc.zero_grad()
    with torch.no_grad():
        _, z_trafico = ae_trafico(Xt_trafico)
        _, z_comportamiento = ae_comportamiento(Xt_comportamiento)
    n = min(len(z_trafico), len(z_comportamiento) * 40)  # balancear tamaños de dominio
    idx_trafico = torch.randperm(len(z_trafico))[:n]
    pred_trafico = discriminador(z_trafico[idx_trafico])
    pred_comportamiento = discriminador(z_comportamiento)
    loss_disc = bce(pred_trafico, torch.ones_like(pred_trafico)) + \
                bce(pred_comportamiento, torch.zeros_like(pred_comportamiento))
    loss_disc.backward()
    opt_disc.step()

    # --- Paso de los autoencoders (reconstrucción + engañar al discriminador) ---
    opt_ae.zero_grad()
    recon_trafico, z_trafico = ae_trafico(Xt_trafico)
    recon_comportamiento, z_comportamiento = ae_comportamiento(Xt_comportamiento)

    loss_recon = mse(recon_trafico, torch.sigmoid(Xt_trafico)) + \
                 mse(recon_comportamiento, torch.sigmoid(Xt_comportamiento))

    pred_trafico_g = discriminador(z_trafico[idx_trafico])
    pred_comportamiento_g = discriminador(z_comportamiento)
    loss_adv = bce(pred_trafico_g, torch.zeros_like(pred_trafico_g)) + \
               bce(pred_comportamiento_g, torch.ones_like(pred_comportamiento_g))

    loss_total = loss_recon + PESO_ADVERSARIAL * loss_adv
    loss_total.backward()
    opt_ae.step()

    if (epoch + 1) % 20 == 0:
        print(f"Época {epoch+1:3d} | recon={loss_recon.item():.4f} | adv={loss_adv.item():.4f} | disc={loss_disc.item():.4f}")


### 6.3 Emparejamiento por similitud coseno y construcción del dataset enriquecido

- Cada embedding de tráfico se empareja con el embedding de comportamiento más similar (coseno) en el espacio latente alineado.
- Se concatena `[embedding_trafico; embedding_comportamiento_emparejado]` para formar el vector enriquecido.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

with torch.no_grad():
    _, emb_trafico = ae_trafico(Xt_trafico)
    _, emb_comportamiento = ae_comportamiento(Xt_comportamiento)

emb_trafico = emb_trafico.numpy()
emb_comportamiento = emb_comportamiento.numpy()

sim = cosine_similarity(emb_trafico, emb_comportamiento)
idx_mas_similar = sim.argmax(axis=1)

X_enriquecido = np.concatenate([emb_trafico, emb_comportamiento[idx_mas_similar]], axis=1)
print("Dataset enriquecido:", X_enriquecido.shape, "(64D tráfico + 32D comportamiento = 96D)")


### 6.4 Clasificación: DFNN con y sin enriquecimiento

- Misma arquitectura para ambos experimentos (3 capas ocultas: 128, 64, 32; ReLU + dropout 0.3), solo cambia el input.
- Se comparan explícitamente para que la diferencia sea el hallazgo, no un detalle secundario.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

class DFNN(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

def entrenar_dfnn(X_in, y_in, epocas=50, batch_size=128, lr=1e-3, verbose_cada=10):
    X_train, X_val, y_train, y_val = train_test_split(X_in, y_in, test_size=0.2,
                                                        stratify=y_in, random_state=42)
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.float32).unsqueeze(1))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    Xv = torch.tensor(X_val, dtype=torch.float32)
    yv = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

    modelo = DFNN(in_dim=X_in.shape[1])
    opt = optim.Adam(modelo.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    historial = {"train_loss": [], "val_loss": []}

    for epoch in range(epocas):
        modelo.train()
        losses = []
        for xb, yb in train_dl:
            opt.zero_grad()
            pred = modelo(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        modelo.eval()
        with torch.no_grad():
            val_pred = modelo(Xv)
            val_loss = loss_fn(val_pred, yv).item()

        historial["train_loss"].append(np.mean(losses))
        historial["val_loss"].append(val_loss)
        if (epoch + 1) % verbose_cada == 0:
            print(f"Época {epoch+1:3d} | train={np.mean(losses):.4f} | val={val_loss:.4f}")

    return modelo, historial, (X_val, y_val)

print("=== Modelo CON enriquecimiento ===")
modelo_enriquecido, hist_enriquecido, val_enriquecido = entrenar_dfnn(X_enriquecido, y_trafico, epocas=50)

print("\n=== Modelo SIN enriquecimiento (baseline, solo tráfico) ===")
modelo_baseline, hist_baseline, val_baseline = entrenar_dfnn(emb_trafico, y_trafico, epocas=50)


## Evaluación

- Curvas de aprendizaje: la brecha train/validación es la señal a observar, no solo el accuracy final.
- Matriz de confusión y métricas estándar para ambos modelos.
- Validación cruzada estratificada (5 folds) sobre el modelo enriquecido, para confirmar que el resultado es estable y no un golpe de suerte del split.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_enriquecido["train_loss"], label="train")
axes[0].plot(hist_enriquecido["val_loss"], label="val")
axes[0].set_title("Con enriquecimiento")
axes[0].legend()
axes[1].plot(hist_baseline["train_loss"], label="train")
axes[1].plot(hist_baseline["val_loss"], label="val")
axes[1].set_title("Sin enriquecimiento (baseline)")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

def evaluar(modelo, X_val, y_val, nombre):
    modelo.eval()
    with torch.no_grad():
        probs = modelo(torch.tensor(X_val, dtype=torch.float32)).numpy().ravel()
    preds = (probs >= 0.5).astype(int)
    print(f"--- {nombre} ---")
    print("Accuracy :", round(accuracy_score(y_val, preds), 4))
    print("Precision:", round(precision_score(y_val, preds), 4))
    print("Recall   :", round(recall_score(y_val, preds), 4))
    print("F1       :", round(f1_score(y_val, preds), 4))
    print("ROC-AUC  :", round(roc_auc_score(y_val, probs), 4))
    print("Matriz de confusión:\n", confusion_matrix(y_val, preds))
    print()

evaluar(modelo_enriquecido, *val_enriquecido, "Con enriquecimiento")
evaluar(modelo_baseline, *val_baseline, "Sin enriquecimiento (baseline)")


In [ ]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas_folds = []

for fold, (idx_tr, idx_te) in enumerate(kf.split(X_enriquecido, y_trafico), start=1):
    modelo_fold, _, _ = entrenar_dfnn(X_enriquecido[idx_tr], y_trafico[idx_tr],
                                       epocas=25, verbose_cada=100)
    modelo_fold.eval()
    with torch.no_grad():
        probs = modelo_fold(torch.tensor(X_enriquecido[idx_te], dtype=torch.float32)).numpy().ravel()
    preds = (probs >= 0.5).astype(int)
    y_te = y_trafico[idx_te]
    metricas_folds.append({
        "fold": fold,
        "accuracy": accuracy_score(y_te, preds),
        "precision": precision_score(y_te, preds),
        "recall": recall_score(y_te, preds),
        "f1": f1_score(y_te, preds),
        "roc_auc": roc_auc_score(y_te, probs),
    })

df_folds = pd.DataFrame(metricas_folds)
print(df_folds.round(4))
print("\nPromedio:\n", df_folds.drop(columns="fold").mean().round(4))


## Hallazgos principales

- El baseline de solo-tráfico llega a **accuracy cercano a 100%** en pocas épocas, con pérdida de entrenamiento y validación cayendo casi juntas a cero: el patrón clásico de memorización, no de generalización.
- El modelo enriquecido converge más lento (30+ épocas), mantiene una **brecha train/validación pequeña pero sostenida**, y su error se reparte de forma balanceada entre falsos positivos y falsos negativos.
- La validación cruzada confirma que el resultado del modelo enriquecido es **estable entre folds**, no depende de un split afortunado.
- El mismo patrón (enriquecido con métricas ligeramente menores pero con evidencia de generalización) se replicó al validar contra un segundo dataset público de tráfico con distribución y ataques distintos, lo que sugiere que el efecto de regularización del enriquecimiento cross-domain no es específico de un solo dataset.
- El hallazgo central del proyecto no fue "subir la métrica", fue diseñar el experimento correcto para saber si una métrica alta era confiable.